In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
meta = pd.read_csv('/projects/fmba_covid/hip_full/metadata.txt', sep='\t').drop(columns=['file_name', '..filter..'])

In [3]:
cmv_positive = meta[(meta.cmv == '+') & (meta.hla.str.contains('A*02')) & (meta.hla.str.contains('B*07'))].reset_index(
    drop=True)

In [4]:
cmv_positive

,sample_id,age,race,sex,cmv,hla
0,HIP00594,21.0,"caucasian,non-hispanic or latino",male,+,"HLA-A*02,HLA-A*32,HLA-B*07,HLA-B*61"
1,HIP00777,22.0,NaN,male,+,"HLA-A*02,HLA-A*03,HLA-B*07,HLA-B*27"
2,HIP03370,41.0,"caucasian,non-hispanic or latino",male,+,"HLA-A*01,HLA-A*02,HLA-B*07,HLA-B*52"
3,HIP03511,48.0,"caucasian,non-hispanic or latino",male,+,"HLA-A*02,HLA-A*23,HLA-B*07,HLA-B*27"
4,HIP03720,38.0,NaN,male,+,"HLA-A*02,HLA-A*26,HLA-B*07,HLA-B*60"
5,HIP04509,26.0,"caucasian,non-hispanic or latino",female,+,"HLA-A*02,HLA-A*03,HLA-B*07,HLA-B*14"
6,HIP05559,21.0,"caucasian,non-hispanic or latino",female,+,"HLA-A*02,HLA-A*03,HLA-B*05,HLA-B*07"
7,HIP05590,49.0,"caucasian,non-hispanic or latino",female,+,"HLA-A*02,HLA-A*23,HLA-B*07,HLA-B*44"
8,HIP05815,NaN,NaN,male,+,"HLA-A*01,HLA-A*02,HLA-B*07,HLA-B*61"
9,HIP05960,NaN,NaN,male,+,"HLA-A*02,HLA-A*11,HLA-B*07"


In [6]:
for sample_name in tqdm(cmv_positive.sample_id):
    df = pd.read_csv(f'/projects/fmba_covid/hip_full/corr/{sample_name}.txt', sep='\t').rename(columns={
        'cdr3aa': 'junction_aa',
        'v': 'v_call', 
        'j': 'j_call',
    })[['junction_aa', 'v_call', 'j_call', 'count']]
    
    df['v_call'] = df.v_call.apply(lambda x: x.split(',')[0])
    df['j_call'] = df.j_call.apply(lambda x: x.split(',')[0])
    df['locus'] = 'beta'
    df = df[df.junction_aa.str.isalpha()]
    if len(df) > 100000:
        df = df[df['count'] > 1].drop(columns=['count'])
        df.to_csv(f'/projects/immunestatus/emerson/airr_format/{sample_name}.tsv', sep='\t', index=False)

100%|██████████| 28/28 [00:26<00:00,  1.04it/s]


In [10]:
cmv_negative = meta[(meta.cmv == '-') & (meta.hla.str.contains('A*02')) & (meta.hla.str.contains('B*07'))].reset_index(drop=True)

In [13]:
cmv_negative = cmv_negative.sort_values(by='hla', ascending=False).head(4)

In [14]:
cmv_negative

,sample_id,age,race,sex,cmv,hla
8,HIP02855,NaN,NaN,female,-,"HLA-A*02,HLA-B*07,HLA-B*60"
9,HIP03484,46.0,"caucasian,non-hispanic or latino",male,-,"HLA-A*02,HLA-B*07,HLA-B*58"
35,HIP14080,28.0,"caucasian,non-hispanic or latino",male,-,"HLA-A*02,HLA-B*07,HLA-B*51"
7,HIP02112,6.0,"caucasian,non-hispanic or latino",female,-,"HLA-A*02,HLA-B*07,HLA-B*15"


In [15]:
control_samples = []
for sample_name in tqdm(cmv_negative.sample_id):
    df = pd.read_csv(f'/projects/fmba_covid/hip_full/corr/{sample_name}.txt', sep='\t').rename(columns={
        'cdr3aa': 'junction_aa',
        'v': 'v_call', 
        'j': 'j_call',
    })[['junction_aa', 'v_call', 'j_call']]
    
    df['v_call'] = df.v_call.apply(lambda x: x.split(',')[0])
    df['j_call'] = df.j_call.apply(lambda x: x.split(',')[0])
    df['locus'] = 'beta'
    df = df[df.junction_aa.str.isalpha()]
    if len(df) > 100000:
        control_samples.append(df)

100%|██████████| 4/4 [00:02<00:00,  1.42it/s]


In [16]:
[len(x) for x in control_samples]

[437969, 118536, 432307]

In [17]:
control_sample = pd.concat(control_samples)

In [18]:
control_sample

,junction_aa,v_call,j_call,locus
0,CASSLGTGPNIQYF,TRBV13,TRBJ2-4,beta
1,CASSLGTGPNIQYF,TRBV13,TRBJ2-4,beta
2,CASRGGYDEAFF,TRBV27,TRBJ1-1,beta
4,CASSWTGSYEQYF,TRBV6-1,TRBJ2-7,beta
5,CASRLGSSEQFF,TRBV6-3,TRBJ2-1,beta
...,...,...,...,...
527010,CATSRDGSYAF,TRBV15,TRBJ1-2,beta
527011,CATSEPDTQYF,TRBV15,TRBJ2-3,beta
527012,CAIGWYNEQFF,TRBV15,TRBJ2-1,beta
527018,CASSESLGSSETQYF,TRBV6-1,TRBJ2-5,beta


In [19]:
control_sample.to_csv('/projects/immunestatus/emerson/airr_format/control.tsv', sep='\t', index=False)